# P4 — Foundation Models cho dự báo nhu cầu gián đoạn

Notebook triển khai ba time-series foundation models trên mẫu M5 gồm **Chronos-Bolt**, **TinyTimeMixer (TTM)** và **Moirai-2**.

Mỗi mô hình được đánh giá theo cùng ba cửa sổ rolling-origin. Với mỗi cửa sổ, mô hình chỉ nhận dữ liệu đến `train_end`, dự báo tiếp qua validation và test, sau đó chỉ giữ 28 ngày test. Đầu ra chuẩn gồm 9 mức quantile: `0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995`.

Phạm vi P4:
- zero-shot forecasting với Chronos-Bolt, TTM và Moirai-2;
- tạo probabilistic forecast phù hợp với từng mô hình;
- kiểm tra schema, quantile crossing và lưu raw forecast;
- benchmark runtime trong cùng môi trường chạy;
- thảo luận rủi ro rò rỉ dữ liệu pre-training.


## 01. Thiết lập môi trường


In [ ]:
from pathlib import Path
import os, platform, random, subprocess, sys, time, warnings

CWD = Path.cwd()
ROOT = CWD if (CWD / "common/config.yaml").exists() else CWD.parent
assert (ROOT / "common/config.yaml").exists(), f"Không xác định được project root từ {CWD}"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)

import numpy as np
import pandas as pd
import torch
import yaml
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Device:", DEVICE)
print("System:", platform.platform())
print("Project root:", ROOT)


## 02. Load cấu hình và dữ liệu


In [ ]:
PATH_CONFIG = ROOT / "common/config.yaml"
PATH_WINDOWS = ROOT / "common/rolling_windows.csv"
PATH_SAMPLE = ROOT / "common/sample_ids.csv"
PATH_LONG = ROOT / "data/processed/data_long.parquet"
PATH_OUTPUT = ROOT / "data/outputs/forecasts"
PATH_OUTPUT.mkdir(parents=True, exist_ok=True)

for name, path in {"config": PATH_CONFIG, "rolling windows": PATH_WINDOWS, "sample ids": PATH_SAMPLE, "data_long": PATH_LONG}.items():
    assert path.exists(), f"Thiếu {name}: {path}"

with open(PATH_CONFIG, encoding="utf-8") as f: CFG = yaml.safe_load(f)

Q = [float(q) for q in CFG["quantiles"]]
H = int(CFG["horizon"])
FM = CFG["foundation"]
windows = pd.read_csv(PATH_WINDOWS)
sample_ids = pd.read_csv(PATH_SAMPLE)
long = pd.read_parquet(PATH_LONG)

print("data_long:", long.shape, "| sample:", sample_ids.shape, "| windows:", len(windows))
display(windows)


## 03. Chuẩn bị dữ liệu và hàm kiểm tra chung


In [ ]:
REQUIRED_COLS = {"series_id", "date", "d_int", "sales", "segment"}
missing = REQUIRED_COLS - set(long.columns)
assert not missing, f"data_long thiếu các cột: {sorted(missing)}"

p4_data = long[["series_id", "date", "d_int", "sales", "segment"]].copy()
p4_data["date"] = pd.to_datetime(p4_data["date"])
p4_data["d_int"] = pd.to_numeric(p4_data["d_int"]).astype(int)
p4_data["sales"] = pd.to_numeric(p4_data["sales"]).astype(np.float32)
p4_data = p4_data.sort_values(["series_id", "d_int"]).reset_index(drop=True)

expected_ids = set(sample_ids["series_id"].astype(str))
actual_ids = set(p4_data["series_id"].astype(str))
assert actual_ids == expected_ids, "series_id trong data_long không khớp sample_ids.csv"
assert p4_data.groupby("d_int")["date"].nunique().max() == 1

day_to_date = p4_data[["d_int", "date"]].drop_duplicates().set_index("d_int")["date"].to_dict()
series_ids = p4_data["series_id"].drop_duplicates().tolist()
series_groups = {sid: g for sid, g in p4_data.groupby("series_id", sort=False)}
N_SERIES = len(series_ids)
COLS = ["series_id", "date", "d_int", "quantile_level", "value", "model"]

print(f"Series: {N_SERIES:,} | d_int: {p4_data['d_int'].min()} → {p4_data['d_int'].max()}")
display(p4_data[["series_id", "segment"]].drop_duplicates()["segment"].value_counts())


In [ ]:
def validate_forecast(df, model_name):
    expected_rows = len(windows) * N_SERIES * H * len(Q)
    assert list(df.columns) == COLS, f"{model_name}: sai schema"
    assert len(df) == expected_rows, f"{model_name}: {len(df):,} rows, expected {expected_rows:,}"
    assert df["series_id"].nunique() == N_SERIES
    assert df["quantile_level"].nunique() == len(Q)
    assert df["value"].notna().all() and np.isfinite(df["value"]).all() and (df["value"] >= 0).all()
    assert set(df["model"].unique()) == {model_name}
    assert df.duplicated(["series_id", "d_int", "quantile_level"]).sum() == 0
    print(f"✓ {model_name}: {len(df):,} rows | {N_SERIES:,} series | {len(Q)} quantiles")

def check_quantile_crossing(df, model_name):
    wide = df.pivot(index=["series_id", "d_int"], columns="quantile_level", values="value")[sorted(Q)]
    n_crossing = int((np.diff(wide.to_numpy(), axis=1) < -1e-8).any(axis=1).sum())
    print(f"✓ {model_name}: quantile crossing = {n_crossing:,}")
    assert n_crossing == 0


## 04. Chronos-Bolt — zero-shot

Chronos-Bolt được dùng trực tiếp từ checkpoint pretrained, không fine-tune trên M5. Context được cắt tại `train_end`; mô hình dự báo từ `train_end + 1` đến `test_end`, sau đó chỉ giữ 28 ngày test.

Chronos-Bolt hỗ trợ quantile native nhưng checkpoint chỉ được huấn luyện cho vùng quantile trung tâm. Khi yêu cầu các mức đuôi ngoài vùng hỗ trợ, thư viện có thể chặn chúng về biên được huấn luyện. Đây là hạn chế cần được giữ nguyên và ghi nhận ở raw forecast thay vì tự ngoại suy các đuôi cực trị.


In [ ]:
from chronos import BaseChronosPipeline

CHRONOS_MODEL = FM["chronos_model"]
CHRONOS_CONTEXT = int(FM["chronos_context"])
CHRONOS_BATCH_SIZE = 64

chronos = BaseChronosPipeline.from_pretrained(CHRONOS_MODEL, device_map=DEVICE, torch_dtype=torch.float32)
print("Chronos-Bolt:", CHRONOS_MODEL, "| context:", CHRONOS_CONTEXT, "| device:", DEVICE)


In [ ]:
chronos_outputs, chronos_runtime = [], []

for w_idx, w in windows.iterrows():
    window_name = str(w["window"]) if "window" in windows.columns else f"W{w_idx + 1}"
    train_end, test_start, test_end = int(w["train_end"]), int(w["test_start"]), int(w["test_end"])
    prediction_length = test_end - train_end
    test_offset = test_start - (train_end + 1)
    test_days = np.arange(test_start, test_end + 1)
    t0, window_parts = time.perf_counter(), []

    for batch_start in range(0, N_SERIES, CHRONOS_BATCH_SIZE):
        batch_ids = series_ids[batch_start:batch_start + CHRONOS_BATCH_SIZE]
        contexts = [torch.tensor(series_groups[sid].loc[series_groups[sid]["d_int"] <= train_end, "sales"].to_numpy(dtype=np.float32)[-CHRONOS_CONTEXT:], dtype=torch.float32) for sid in batch_ids]
        context_batch = torch.stack(contexts)

        with torch.inference_mode():
            q_pred, _ = chronos.predict_quantiles(inputs=context_batch, prediction_length=prediction_length, quantile_levels=Q)

        q_test = np.clip(q_pred.detach().cpu().numpy()[:, test_offset:test_offset + H, :], 0, None)
        assert q_test.shape == (len(batch_ids), H, len(Q))

        for i, sid in enumerate(batch_ids):
            block = pd.DataFrame({"series_id": np.repeat(sid, H * len(Q)), "d_int": np.repeat(test_days, len(Q)), "quantile_level": np.tile(Q, H), "value": q_test[i].reshape(-1)})
            block["date"] = block["d_int"].map(day_to_date); block["model"] = "chronos_bolt"
            window_parts.append(block[COLS])

    window_df = pd.concat(window_parts, ignore_index=True)
    elapsed = time.perf_counter() - t0
    chronos_outputs.append(window_df)
    chronos_runtime.append({"model": "chronos_bolt", "window": window_name, "seconds": elapsed})
    print(f"{window_name}: {len(window_df):,} rows | {elapsed:.1f} s")

chronos_all = pd.concat(chronos_outputs, ignore_index=True)
chronos_all["quantile_level"] = chronos_all["quantile_level"].astype(float)
chronos_all["value"] = chronos_all["value"].astype(np.float32)
validate_forecast(chronos_all, "chronos_bolt")
check_quantile_crossing(chronos_all, "chronos_bolt")

chronos_all.to_parquet(PATH_OUTPUT / "chronos_bolt.parquet", index=False)
pd.DataFrame(chronos_runtime).to_csv(PATH_OUTPUT / "chronos_bolt_runtime.csv", index=False)


## 05. TinyTimeMixer (TTM) — zero-shot

TTM được dùng ở chế độ zero-shot và trả về point forecast. Mỗi context được chuẩn hóa theo mean/std tính **chỉ từ lịch sử đến `train_end`**, sau đó inverse-transform dự báo.

Để tạo 9 quantile theo schema chung, point forecast được bọc bằng phân phối Negative Binomial khi phương sai lịch sử lớn hơn trung bình; các trường hợp còn lại dùng Poisson. Vì vậy probabilistic forecast của TTM trong notebook này là **hậu xử lý**, không phải quantile native của checkpoint.


In [ ]:
from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction

TTM_MODEL = FM["ttm_model"]
TTM_CONTEXT = int(FM["ttm_context"])
TTM_BATCH_SIZE = 64

ttm = TinyTimeMixerForPrediction.from_pretrained(TTM_MODEL, num_input_channels=1).to(DEVICE)
ttm.eval()
print("TTM:", TTM_MODEL, "| context:", TTM_CONTEXT, "| device:", DEVICE)


In [ ]:
def get_dispersion(data, train_end):
    mv = data.loc[data["d_int"] <= train_end].groupby("series_id")["sales"].agg(["mean", "var"])
    mean, var = mv["mean"].to_numpy(), mv["var"].to_numpy()
    r = np.full(len(mv), np.inf, dtype=np.float64)
    use_nb = (mean > 0) & (var > mean)
    r[use_nb] = mean[use_nb] ** 2 / (var[use_nb] - mean[use_nb])
    return dict(zip(mv.index, r))

def point_to_quantiles(point_df, model_name, disp_r):
    pf = point_df.copy()
    pf["value"] = pf["value"].clip(lower=0).astype(float)
    pf["r"] = pf["series_id"].map(disp_r).fillna(np.inf)
    mu, r = pf["value"].to_numpy(), pf["r"].to_numpy()
    finite = np.isfinite(r) & (r > 0)
    outputs = []

    for q in Q:
        qv = np.empty_like(mu, dtype=float)
        if finite.any():
            rr, mm = r[finite], mu[finite]
            p = rr / (rr + np.where(mm > 0, mm, 1e-9))
            qv[finite] = stats.nbinom.ppf(q, rr, p)
        if (~finite).any(): qv[~finite] = stats.poisson.ppf(q, mu[~finite])

        block = pf[["series_id", "date", "d_int"]].copy()
        block["quantile_level"] = float(q)
        block["value"] = np.nan_to_num(qv, nan=0.0, posinf=0.0, neginf=0.0)
        block["model"] = model_name
        outputs.append(block)

    return pd.concat(outputs, ignore_index=True)[COLS]


In [ ]:
ttm_outputs, ttm_runtime = [], []

for w_idx, w in windows.iterrows():
    window_name = str(w["window"]) if "window" in windows.columns else f"W{w_idx + 1}"
    train_end, test_start, test_end = int(w["train_end"]), int(w["test_start"]), int(w["test_end"])
    prediction_length = test_end - train_end
    test_offset = test_start - (train_end + 1)
    test_days = np.arange(test_start, test_end + 1)
    disp_r = get_dispersion(p4_data, train_end)
    t0, window_points = time.perf_counter(), []

    for batch_start in range(0, N_SERIES, TTM_BATCH_SIZE):
        batch_ids = series_ids[batch_start:batch_start + TTM_BATCH_SIZE]
        contexts, means, stds = [], [], []

        for sid in batch_ids:
            y = series_groups[sid].loc[series_groups[sid]["d_int"] <= train_end, "sales"].to_numpy(dtype=np.float32)[-TTM_CONTEXT:]
            mean, std = float(y.mean()), float(y.std())
            if std < 1e-6: std = 1.0
            contexts.append((y - mean) / std); means.append(mean); stds.append(std)

        past_values = torch.tensor(np.stack(contexts), dtype=torch.float32).unsqueeze(-1).to(DEVICE)
        with torch.inference_mode(): outputs = ttm(past_values=past_values)

        point_scaled = outputs.prediction_outputs.detach().cpu().numpy()[..., 0]
        point_pred = point_scaled * np.asarray(stds, dtype=np.float32)[:, None] + np.asarray(means, dtype=np.float32)[:, None]
        point_test = np.clip(point_pred[:, :prediction_length], 0, None)[:, test_offset:test_offset + H]
        assert point_test.shape == (len(batch_ids), H)

        batch_df = pd.DataFrame({"series_id": np.repeat(batch_ids, H), "d_int": np.tile(test_days, len(batch_ids)), "value": point_test.reshape(-1)})
        batch_df["date"] = batch_df["d_int"].map(day_to_date)
        window_points.append(batch_df)

    point_window = pd.concat(window_points, ignore_index=True)
    window_df = point_to_quantiles(point_window, "ttm", disp_r)
    elapsed = time.perf_counter() - t0
    ttm_outputs.append(window_df)
    ttm_runtime.append({"model": "ttm", "window": window_name, "seconds": elapsed})
    print(f"{window_name}: {len(window_df):,} rows | {elapsed:.1f} s")

ttm_all = pd.concat(ttm_outputs, ignore_index=True)
ttm_all["quantile_level"] = ttm_all["quantile_level"].astype(float)
ttm_all["value"] = ttm_all["value"].astype(np.float32)
validate_forecast(ttm_all, "ttm")
check_quantile_crossing(ttm_all, "ttm")

ttm_all.to_parquet(PATH_OUTPUT / "ttm.parquet", index=False)
pd.DataFrame(ttm_runtime).to_csv(PATH_OUTPUT / "ttm_runtime.csv", index=False)


## 06. Moirai-2 — zero-shot

**LƯU Ý:** Moirai-2 được chạy trong môi trường Python riêng biệt vì phiên bản Uni2TS yêu cầu dependency khác với môi trường chính của Chronos/TTM.

Checkpoint `Salesforce/moirai-2.0-R-small` cung cấp 9 quantile native từ 0.1 đến 0.9. Các mức yêu cầu nằm trong khoảng này được nội suy tuyến tính; các mức ngoài khoảng được chặn tại biên 0.1/0.9. Sau mapping, **quantile rearrangement** sắp xếp các giá trị tại từng `(series_id, d_int)` để loại quantile crossing. Đây là bước hậu xử lý, không phải output native của Moirai-2.


In [ ]:
MOIRAI_ENV = ROOT / ".venv-moirai"
MOIRAI_PYTHON = MOIRAI_ENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

if not MOIRAI_PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(MOIRAI_ENV)], check=True)
    subprocess.run([str(MOIRAI_PYTHON), "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
    subprocess.run([str(MOIRAI_PYTHON), "-m", "pip", "install", "-q", "uni2ts==2.0.0", "pyarrow"], check=True)

print("Moirai Python:", MOIRAI_PYTHON)


In [ ]:
moirai_code = '\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport pandas as pd\nfrom gluonts.dataset.common import ListDataset\nfrom uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module\n\nROOT = Path.cwd()\nMODEL = "Salesforce/moirai-2.0-R-small"\nCONTEXT, BATCH_SIZE, H = 512, 64, 28\nQ = np.array([0.005, 0.025, 0.165, 0.25, 0.5, 0.75, 0.835, 0.975, 0.995])\nNATIVE_Q = np.arange(0.1, 1.0, 0.1)\nCOLS = ["series_id", "date", "d_int", "quantile_level", "value", "model"]\n\ndata = pd.read_parquet(ROOT / "data/processed/data_long.parquet")[["series_id", "date", "d_int", "sales"]].copy()\ndata["date"] = pd.to_datetime(data["date"]); data["d_int"] = data["d_int"].astype(int); data["sales"] = data["sales"].astype("float32")\ndata = data.sort_values(["series_id", "d_int"]).reset_index(drop=True)\ngroups = {sid: g for sid, g in data.groupby("series_id", sort=False)}\nids = list(groups)\ndates = data[["d_int", "date"]].drop_duplicates()\nwindows = pd.read_csv(ROOT / "common/rolling_windows.csv")\nmodule = Moirai2Module.from_pretrained(MODEL)\n\nall_outputs, runtime_rows = [], []\n\nfor _, w in windows.iterrows():\n    window = str(w["window"])\n    train_end, test_start, test_end = int(w["train_end"]), int(w["test_start"]), int(w["test_end"])\n    pred_len = test_end - train_end\n    t0 = time.perf_counter()\n\n    entries = []\n    for sid in ids:\n        x = groups[sid].loc[groups[sid]["d_int"] <= train_end].tail(CONTEXT)\n        assert len(x) == CONTEXT\n        entries.append({"start": pd.Period(x["date"].iloc[0], freq="D"), "target": x["sales"].to_numpy(dtype=np.float32), "item_id": sid})\n\n    dataset = ListDataset(entries, freq="D")\n    model = Moirai2Forecast(prediction_length=pred_len, target_dim=1, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0, context_length=CONTEXT, module=module)\n    forecasts = list(model.create_predictor(batch_size=BATCH_SIZE, device="auto").predict(dataset))\n\n    keep_from = test_start - (train_end + 1)\n    test_days = np.arange(test_start, test_end + 1)\n    rows = []\n\n    for sid, fcst in zip(ids, forecasts):\n        native = np.maximum(fcst.forecast_array[:, keep_from:keep_from + H], 0)\n        assert native.shape == (len(NATIVE_Q), H)\n        for j, d_int in enumerate(test_days):\n            values = np.sort(np.interp(Q, NATIVE_Q, native[:, j]))\n            rows.extend((sid, d_int, float(q), float(value)) for q, value in zip(Q, values))\n\n    out = pd.DataFrame(rows, columns=["series_id", "d_int", "quantile_level", "value"]).merge(dates, on="d_int", how="left")\n    out["model"] = "moirai_2"; out = out[COLS]\n    elapsed = time.perf_counter() - t0\n    all_outputs.append(out); runtime_rows.append({"model": "moirai_2", "window": window, "seconds": elapsed})\n    print(f"{window}: {len(out):,} rows | {elapsed:.1f} s")\n\nresult = pd.concat(all_outputs, ignore_index=True)\nexpected_rows = len(windows) * len(ids) * H * len(Q)\nassert len(result) == expected_rows and result["series_id"].nunique() == len(ids)\nassert result["value"].notna().all() and np.isfinite(result["value"]).all() and (result["value"] >= 0).all()\n\nwide = result.pivot(index=["series_id", "d_int"], columns="quantile_level", values="value")[sorted(Q)]\nassert (np.diff(wide.to_numpy(), axis=1) < -1e-8).any(axis=1).sum() == 0\n\noutput_dir = ROOT / "data/outputs/forecasts"\noutput_dir.mkdir(parents=True, exist_ok=True)\nresult.to_parquet(output_dir / "moirai_2.parquet", index=False)\npd.DataFrame(runtime_rows).to_csv(output_dir / "moirai_2_runtime.csv", index=False)\nprint("✓ Moirai-2:", result.shape)\n'

result_moirai = subprocess.run([str(MOIRAI_PYTHON), "-c", moirai_code], cwd=ROOT, capture_output=True, text=True)
print(result_moirai.stdout)
if result_moirai.returncode != 0: print(result_moirai.stderr)
assert result_moirai.returncode == 0, "Moirai-2 full inference thất bại"


In [ ]:
moirai = pd.read_parquet(PATH_OUTPUT / "moirai_2.parquet")
validate_forecast(moirai, "moirai_2")
check_quantile_crossing(moirai, "moirai_2")

wide = moirai.pivot(index=["series_id", "d_int"], columns="quantile_level", values="value")
print("q0.005 = q0.025:", f"{np.isclose(wide[0.005], wide[0.025]).mean():.2%}")
print("q0.975 = q0.995:", f"{np.isclose(wide[0.975], wide[0.995]).mean():.2%}")


## 07. Benchmark thời gian chạy

Runtime chỉ được so sánh khi cả ba mô hình được chạy trong **cùng một phiên notebook, cùng phần cứng, cùng 3.474 chuỗi và cùng ba cửa sổ rolling-origin**. Các con số này mô tả chi phí tính toán trong môi trường thực nghiệm hiện tại, không phải benchmark tốc độ tổng quát giữa các kiến trúc.


In [ ]:
runtime = pd.concat([
    pd.read_csv(PATH_OUTPUT / "chronos_bolt_runtime.csv"),
    pd.read_csv(PATH_OUTPUT / "ttm_runtime.csv"),
    pd.read_csv(PATH_OUTPUT / "moirai_2_runtime.csv")
], ignore_index=True)

runtime_summary = runtime.groupby("model", as_index=False)["seconds"].agg(total_seconds="sum", avg_seconds_per_window="mean")
runtime_summary["total_minutes"] = runtime_summary["total_seconds"] / 60
runtime_summary = runtime_summary.sort_values("total_seconds").reset_index(drop=True)

display(runtime)
display(runtime_summary)


## 08. Rủi ro rò rỉ dữ liệu pre-training

Một vấn đề quan trọng khi đánh giá foundation model theo thiết lập zero-shot là khả năng dữ liệu đánh giá đã xuất hiện trong pre-training. Vì vậy, **zero-shot trong thí nghiệm này chỉ có nghĩa là không cập nhật trọng số bằng dữ liệu M5**, không đồng nghĩa tuyệt đối với *unseen during pre-training*.

### Chronos-Bolt

Model card Chronos-Bolt cho biết mô hình được huấn luyện trên gần 100 tỷ quan sát và các benchmark dùng trong đánh giá zero-shot của tác giả không được đưa vào training. Hệ sinh thái dữ liệu Chronos công khai có chứa M5, nhưng repository này bao gồm dữ liệu phục vụ cả training và evaluation. Vì vậy, sự hiện diện của M5 trong repository không đủ để kết luận checkpoint Chronos-Bolt đã được pre-train trên M5.

Nguồn: https://huggingface.co/amazon/chronos-bolt-base

### TinyTimeMixer

Model card của checkpoint TTM r2 công bố các nhóm dữ liệu pre-training và không liệt kê M5. Dựa trên thông tin công khai hiện có, không có bằng chứng cho thấy checkpoint TTM r2 sử dụng trực tiếp M5 trong pre-training. Tuy nhiên, đây không phải bằng chứng tuyệt đối rằng mọi dạng trùng lặp dữ liệu đều không tồn tại.

Nguồn: https://huggingface.co/ibm-granite/granite-timeseries-ttm-r2

### Moirai-2

Model card Moirai-2 mô tả pre-training bằng các phần không gây rò rỉ của GIFT-Eval, dữ liệu mixup từ các phần không gây rò rỉ của Chronos Dataset, dữ liệu tổng hợp và dữ liệu vận hành nội bộ của Salesforce. Tài liệu công khai vì vậy không cung cấp cơ sở để khẳng định Moirai-2 đã được pre-train trực tiếp trên phần M5 dùng trong nghiên cứu này.

Nguồn: https://huggingface.co/Salesforce/moirai-2.0-R-small

### Cách diễn giải

Hiệu năng tốt trên M5 không được xem là bằng chứng rằng mô hình đã ghi nhớ M5. Ngược lại, việc chạy zero-shot cũng không chứng minh tuyệt đối rằng không tồn tại contamination. Kết quả Foundation Models vì vậy cần được diễn giải cùng với giới hạn provenance của các corpus pre-training quy mô lớn.


## 09. Kiểm tra đầu ra cuối

Ba file raw forecast được xuất theo cùng schema 6 cột để chuyển cho bước hiệu chuẩn và đánh giá tiếp theo. Chronos giữ nguyên hạn chế tail clipping của checkpoint; TTM dùng NB/Poisson wrapper; Moirai-2 dùng quantile mapping và rearrangement để bảo đảm non-crossing.


In [ ]:
forecast_files = {"chronos_bolt": PATH_OUTPUT / "chronos_bolt.parquet", "ttm": PATH_OUTPUT / "ttm.parquet", "moirai_2": PATH_OUTPUT / "moirai_2.parquet"}

for model_name, path in forecast_files.items():
    df = pd.read_parquet(path)
    validate_forecast(df, model_name)
    check_quantile_crossing(df, model_name)
    print(f"{model_name:<14} {path.name:<24} {path.stat().st_size / 1024**2:.1f} MB")

print("\n✓ P4 hoàn tất: 3 Foundation Models đã có raw forecast hợp lệ.")
